In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict

# ===============================
# PATHS
# ===============================
ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad"
de_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed"
pt_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_pseudotime/pseudotime_centroid/publication_summary_statistics.csv"

cell_types = ["Ast", "Mic", "Inh", "Oli", "Opc", "Ex"]

de_files = {
    "Ast": "poisson_DE_results_SEAAD_Ast_COMBINED.csv",
    "Mic": "poisson_DE_results_SEAAD_Mic_COMBINED.csv",
    "Inh": "poisson_DE_results_SEAAD_Inh_COMBINED.csv",
    "Oli": "poisson_DE_results_SEAAD_Oli_COMBINED.csv",
    "Opc": "poisson_DE_results_SEAAD_Opc_COMBINED.csv",
    "Ex":  "poisson_DE_results_SEAAD_Ex_COMBINED.csv"
}

# SEA-AD pseudotime subclusters use 'In' prefix while ML/DEG use 'Inh'.
# This map converts cell-type label -> the prefix used in the Subcluster column.
pt_prefix_map = {"Ast": "Ast", "Mic": "Mic", "Inh": "In", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}

# ===============================
# 1. ML PREDICTORS
# ===============================
ml_genes = defaultdict(set)

for ct in cell_types:
    gene_counts = defaultdict(int)

    for split in range(1, 6):
        path = os.path.join(ml_base, ct, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue

        model = joblib.load(path)
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts[g] += 1

    for g, c in gene_counts.items():
        if c >= 2:
            ml_genes[ct].add(g)

# ===============================
# 2. DEG GENES
# ===============================
deg_genes = {}

for ct in cell_types:
    df = pd.read_csv(os.path.join(de_base, de_files[ct]))
    df = df[(df["p_adj"] < 0.05) & (df["log2FC"].abs() > 0.25)]
    deg_genes[ct] = set(df["gene"])

# ===============================
# 3. PSEUDOTIME GENES (TOP 20, AGGREGATED BY CELL TYPE)
# ===============================
pt_df = pd.read_csv(pt_path)

pt_df["top_trajectory_genes"] = pt_df["top_trajectory_genes"].fillna("").apply(
    lambda x: [g.strip() for g in x.split(",")]
)

pt_genes = defaultdict(set)

def infer_cell_type_from_subcluster(subcluster):
    for ct in cell_types:
        if subcluster.startswith(pt_prefix_map[ct]):
            return ct
    return None

for _, row in pt_df.iterrows():
    subcluster = row["Subcluster"]
    ct = infer_cell_type_from_subcluster(subcluster)
    if ct is None:
        continue

    top20 = row["top_trajectory_genes"][:20]
    pt_genes[ct].update(top20)

# ===============================
# TRI-SUPPORTED GENES
# ===============================
tri_supported = {}



print("\n=== TRI-SUPPORTED GENE COUNTS ===")
for ct in cell_types:
    tri = ml_genes[ct] & deg_genes[ct] & pt_genes[ct]
    tri_supported[ct] = tri
    print(f"{ct}: {len(tri)} genes")

# Clean gene name types
for ct in tri_supported:
    tri_supported[ct] = set(str(g) for g in tri_supported[ct])

print("\n=== TRI-SUPPORTED GENES (PER CELL TYPE) ===")
for ct, genes in tri_supported.items():
    print(f"\n{ct}:")
    print(sorted(genes))



=== TRI-SUPPORTED GENE COUNTS ===
Ast: 1 genes
Mic: 2 genes
Inh: 0 genes
Oli: 0 genes
Opc: 1 genes
Ex: 0 genes

=== TRI-SUPPORTED GENES (PER CELL TYPE) ===

Ast:
['SHISA6']

Mic:
['PCDH9', 'RPL7A']

Inh:
[]

Oli:
[]

Opc:
['HSP90AA1']

Ex:
[]


In [2]:
import os
import pandas as pd
import joblib
from collections import defaultdict

# ===============================
# PATHS — ROSMAP
# ===============================
rosmap_ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
rosmap_de_base = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
rosmap_pt_path = "/n/groups/patel/adithya/Pseudotime_Outputs_HVG_final/publication_summary_statistics.csv"

rosmap_de_files = {
    "Ast": "poisson_DE_results_Ast.csv",
    "Mic": "poisson_DE_results_Mic.csv",
    "In":  "poisson_DE_results_In.csv",
    "Oli": "poisson_DE_results_Oli.csv",
    "Opc": "poisson_DE_results_Opc.csv",
    "Ex":  "poisson_DE_results_Ex.csv",
}

# ===============================
# PATHS — SEA-AD
# ===============================
seaad_ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad"
seaad_de_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed"
seaad_pt_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_pseudotime/pseudotime_centroid/publication_summary_statistics.csv"

seaad_de_files = {
    "Ast": "poisson_DE_results_SEAAD_Ast_COMBINED.csv",
    "Mic": "poisson_DE_results_SEAAD_Mic_COMBINED.csv",
    "Inh": "poisson_DE_results_SEAAD_Inh_COMBINED.csv",
    "Oli": "poisson_DE_results_SEAAD_Oli_COMBINED.csv",
    "Opc": "poisson_DE_results_SEAAD_Opc_COMBINED.csv",
    "Ex":  "poisson_DE_results_SEAAD_Ex_COMBINED.csv",
}

# ===============================
# CELL-TYPE NAMING ACROSS DATASETS
# ===============================
# We iterate on a canonical label (SEA-AD style). For each layer we look up
# the matching label/prefix used in ROSMAP vs SEAAD.
canonical_cell_types = ["Ast", "Mic", "Inh", "Oli", "Opc", "Ex"]

# ML/DEG: ROSMAP uses "In", SEAAD uses "Inh"
rosmap_ct_map = {"Ast": "Ast", "Mic": "Mic", "Inh": "In", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}
seaad_ct_map  = {"Ast": "Ast", "Mic": "Mic", "Inh": "Inh", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}

# Pseudotime Subcluster prefixes: BOTH datasets use 'In' for inhibitory.
rosmap_pt_prefix = {"Ast": "Ast", "Mic": "Mic", "Inh": "In", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}
seaad_pt_prefix  = {"Ast": "Ast", "Mic": "Mic", "Inh": "In", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}


# ===============================
# Loaders (mirror figure_1.ipynb logic)
# ===============================
def load_ml_predictors(ml_base, cell_type_in_folder):
    gene_counts = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(ml_base, cell_type_in_folder, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts[g] += 1
    return {g for g, c in gene_counts.items() if c >= 2}

def load_degs(de_base, fname):
    df = pd.read_csv(os.path.join(de_base, fname))
    df = df[(df["p_adj"] < 0.05) & (df["log2FC"].abs() > 0.25)]
    return set(df["gene"].astype(str))

def load_pseudotime_top20(pt_path, prefix_for_ct):
    pt_df = pd.read_csv(pt_path)
    pt_df["top_trajectory_genes"] = pt_df["top_trajectory_genes"].fillna("").apply(
        lambda x: [g.strip() for g in x.split(",")]
    )
    pt_genes = defaultdict(set)
    for _, row in pt_df.iterrows():
        sub = row["Subcluster"]
        for ct, pref in prefix_for_ct.items():
            if sub.startswith(pref):
                pt_genes[ct].update(row["top_trajectory_genes"][:20])
                break
    return pt_genes


# ===============================
# Build sets for both datasets
# ===============================
rosmap_ml  = {ct: load_ml_predictors(rosmap_ml_base, rosmap_ct_map[ct]) for ct in canonical_cell_types}
seaad_ml   = {ct: load_ml_predictors(seaad_ml_base,  seaad_ct_map[ct])  for ct in canonical_cell_types}

rosmap_deg = {ct: load_degs(rosmap_de_base, rosmap_de_files[rosmap_ct_map[ct]]) for ct in canonical_cell_types}
seaad_deg  = {ct: load_degs(seaad_de_base,  seaad_de_files[seaad_ct_map[ct]])  for ct in canonical_cell_types}

rosmap_pt  = load_pseudotime_top20(rosmap_pt_path, rosmap_pt_prefix)
seaad_pt   = load_pseudotime_top20(seaad_pt_path,  seaad_pt_prefix)


# ===============================
# Print per-cell-type overlap for each layer
# ===============================
def overlap_block(layer_name, rosmap_sets, seaad_sets):
    print(f"\n=== {layer_name}: ROSMAP ∩ SEA-AD per cell type ===")
    print(f"{'cell_type':<6} {'ROSMAP':>8} {'SEA-AD':>8} {'overlap':>8}")
    for ct in canonical_cell_types:
        r = rosmap_sets.get(ct, set())
        s = seaad_sets.get(ct, set())
        ov = r & s
        print(f"{ct:<6} {len(r):>8} {len(s):>8} {len(ov):>8}")
        if ov:
            print(f"        overlap genes: {sorted(ov)[:30]}{' ...' if len(ov) > 30 else ''}")

overlap_block("ML predictors (>=2/5 splits)", rosmap_ml,  seaad_ml)
overlap_block("DEGs (p_adj<0.05 & |log2FC|>0.25)", rosmap_deg, seaad_deg)
overlap_block("Pseudotime top-20 (aggregated)", rosmap_pt, seaad_pt)



=== ML predictors (>=2/5 splits): ROSMAP ∩ SEA-AD per cell type ===
cell_type   ROSMAP   SEA-AD  overlap
Ast         175       40        7
        overlap genes: [np.str_('ARL17B'), np.str_('FTH1'), np.str_('KCNIP4'), np.str_('MRPS6'), np.str_('SLC14A1'), np.str_('SYTL4'), np.str_('UTY')]
Mic         466      559       81
        overlap genes: [np.str_('A2M'), np.str_('ABR'), np.str_('ADGRB3'), np.str_('APOE'), np.str_('ARHGAP26'), np.str_('ARL17B'), np.str_('ATP8B4'), np.str_('B3GNT5'), np.str_('BIN1'), np.str_('BMP2K'), np.str_('CACNA1A'), np.str_('CARD11'), np.str_('CDK6'), np.str_('CELF2'), np.str_('CHKA'), np.str_('CPVL'), np.str_('CSF2RA'), np.str_('CTSB'), np.str_('CYFIP1'), np.str_('DOCK4'), np.str_('DOCK8'), np.str_('DPYD'), np.str_('EPB41L2'), np.str_('EPB41L3'), np.str_('FAU'), np.str_('FCGBP'), np.str_('FCHSD2'), np.str_('FKBP5'), np.str_('FMN1'), np.str_('FOXN3')] ...
Inh         108       24        2
        overlap genes: [np.str_('ARL17B'), np.str_('TARBP1')]
Oli     

In [1]:
# Identical to above but with significance values
import os
import pandas as pd
import joblib
from collections import defaultdict
from scipy.stats import fisher_exact

# ===============================
# PATHS — ROSMAP
# ===============================
rosmap_ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
rosmap_de_base = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
rosmap_pt_path = "/n/groups/patel/adithya/Pseudotime_Outputs_HVG_final/publication_summary_statistics.csv"

rosmap_de_files = {
    "Ast": "poisson_DE_results_Ast.csv",
    "Mic": "poisson_DE_results_Mic.csv",
    "In":  "poisson_DE_results_In.csv",
    "Oli": "poisson_DE_results_Oli.csv",
    "Opc": "poisson_DE_results_Opc.csv",
    "Ex":  "poisson_DE_results_Ex.csv",
}

# ===============================
# PATHS — SEA-AD
# ===============================
seaad_ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad"
seaad_de_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed"
seaad_pt_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_pseudotime/pseudotime_centroid/publication_summary_statistics.csv"

seaad_de_files = {
    "Ast": "poisson_DE_results_SEAAD_Ast_COMBINED.csv",
    "Mic": "poisson_DE_results_SEAAD_Mic_COMBINED.csv",
    "Inh": "poisson_DE_results_SEAAD_Inh_COMBINED.csv",
    "Oli": "poisson_DE_results_SEAAD_Oli_COMBINED.csv",
    "Opc": "poisson_DE_results_SEAAD_Opc_COMBINED.csv",
    "Ex":  "poisson_DE_results_SEAAD_Ex_COMBINED.csv",
}

# ===============================
# CELL-TYPE NAMING ACROSS DATASETS
# ===============================
canonical_cell_types = ["Ast", "Mic", "Inh", "Oli", "Opc", "Ex"]
rosmap_ct_map = {"Ast": "Ast", "Mic": "Mic", "Inh": "In", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}
seaad_ct_map  = {"Ast": "Ast", "Mic": "Mic", "Inh": "Inh", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}
rosmap_pt_prefix = {"Ast": "Ast", "Mic": "Mic", "Inh": "In", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}
seaad_pt_prefix  = {"Ast": "Ast", "Mic": "Mic", "Inh": "In", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}


# ===============================
# Loaders (unchanged)
# ===============================
def load_ml_predictors(ml_base, cell_type_in_folder):
    gene_counts = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(ml_base, cell_type_in_folder, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts[g] += 1
    return {g for g, c in gene_counts.items() if c >= 2}

def load_degs(de_base, fname):
    df = pd.read_csv(os.path.join(de_base, fname))
    df = df[(df["p_adj"] < 0.05) & (df["log2FC"].abs() > 0.25)]
    return set(df["gene"].astype(str))

def load_pseudotime_top20(pt_path, prefix_for_ct):
    pt_df = pd.read_csv(pt_path)
    pt_df["top_trajectory_genes"] = pt_df["top_trajectory_genes"].fillna("").apply(
        lambda x: [g.strip() for g in x.split(",")]
    )
    pt_genes = defaultdict(set)
    for _, row in pt_df.iterrows():
        sub = row["Subcluster"]
        for ct, pref in prefix_for_ct.items():
            if sub.startswith(pref):
                pt_genes[ct].update(row["top_trajectory_genes"][:20])
                break
    return pt_genes


# ===============================
# Build sets
# ===============================
rosmap_ml  = {ct: load_ml_predictors(rosmap_ml_base, rosmap_ct_map[ct]) for ct in canonical_cell_types}
seaad_ml   = {ct: load_ml_predictors(seaad_ml_base,  seaad_ct_map[ct])  for ct in canonical_cell_types}

rosmap_deg = {ct: load_degs(rosmap_de_base, rosmap_de_files[rosmap_ct_map[ct]]) for ct in canonical_cell_types}
seaad_deg  = {ct: load_degs(seaad_de_base,  seaad_de_files[seaad_ct_map[ct]])  for ct in canonical_cell_types}

rosmap_pt  = load_pseudotime_top20(rosmap_pt_path, rosmap_pt_prefix)
seaad_pt   = load_pseudotime_top20(seaad_pt_path,  seaad_pt_prefix)


# ===============================
# Background universe (same parquet you used in earlier Fisher blocks)
# ===============================
gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet')
background_genes = set(gene_matrix.index.astype(str))

# Belt-and-suspenders: union in any gene seen in either dataset so d >= 0.
all_seen = set()
for d in (rosmap_ml, seaad_ml, rosmap_deg, seaad_deg, rosmap_pt, seaad_pt):
    for s in d.values():
        all_seen.update(s)
background_genes |= all_seen
N_BG = len(background_genes)


# ===============================
# Fisher per cell type, per layer
# ===============================
def overlap_block(layer_name, rosmap_sets, seaad_sets):
    print(f"\n=== {layer_name}: ROSMAP ∩ SEA-AD per cell type ===")
    print(f"    Background universe size: {N_BG:,}")
    print(f"{'cell_type':<6} {'ROSMAP':>8} {'SEA-AD':>8} {'overlap':>8} {'OR':>10} {'p_value':>12}")
    for ct in canonical_cell_types:
        r = rosmap_sets.get(ct, set())
        s = seaad_sets.get(ct, set())
        ov = r & s

        a = len(ov)
        b = len(r - ov)                       # ROSMAP only
        c = len(s - ov)                       # SEAAD only
        d = N_BG - (a + b + c)                # Background not in either

        if d < 0:
            raise ValueError(f"Negative d for {layer_name}/{ct}: a={a} b={b} c={c} d={d}")

        odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative='greater')

        print(f"{ct:<6} {len(r):>8} {len(s):>8} {len(ov):>8} {odds_ratio:>10.3f} {p_value:>12.3e}")
        if ov:
            print(f"        overlap genes: {sorted(ov)[:30]}{' ...' if len(ov) > 30 else ''}")

overlap_block("ML predictors (>=2/5 splits)", rosmap_ml,  seaad_ml)
overlap_block("DEGs (p_adj<0.05 & |log2FC|>0.25)", rosmap_deg, seaad_deg)
overlap_block("Pseudotime top-20 (aggregated)", rosmap_pt, seaad_pt)



=== ML predictors (>=2/5 splits): ROSMAP ∩ SEA-AD per cell type ===
    Background universe size: 19,011
cell_type   ROSMAP   SEA-AD  overlap         OR      p_value
Ast         175       40        7     23.741    7.167e-08
        overlap genes: [np.str_('ARL17B'), np.str_('FTH1'), np.str_('KCNIP4'), np.str_('MRPS6'), np.str_('SLC14A1'), np.str_('SYTL4'), np.str_('UTY')]
Mic         466      559       81      7.952    1.862e-39
        overlap genes: [np.str_('A2M'), np.str_('ABR'), np.str_('ADGRB3'), np.str_('APOE'), np.str_('ARHGAP26'), np.str_('ARL17B'), np.str_('ATP8B4'), np.str_('B3GNT5'), np.str_('BIN1'), np.str_('BMP2K'), np.str_('CACNA1A'), np.str_('CARD11'), np.str_('CDK6'), np.str_('CELF2'), np.str_('CHKA'), np.str_('CPVL'), np.str_('CSF2RA'), np.str_('CTSB'), np.str_('CYFIP1'), np.str_('DOCK4'), np.str_('DOCK8'), np.str_('DPYD'), np.str_('EPB41L2'), np.str_('EPB41L3'), np.str_('FAU'), np.str_('FCGBP'), np.str_('FCHSD2'), np.str_('FKBP5'), np.str_('FMN1'), np.str_('FOXN3')]